# Trigger `source_to_s3_migration` DAG — End-to-End Test Harness

One-shot setup + trigger for the MapR-to-S3 migration DAG against the shared MapR edge node
running in `hadoop-ssh` namespace. Run cells top-to-bottom on a fresh JupyterHub kernel.

### What this notebook does

| Step | Action |
|---|---|
| 0 | Verify the `hadoop-edge` pod is running and SSH/Hive ports are listening |
| 1 | Populate HDFS + Hive metastore with test data (`setup-test-data.sh`) |
| 2 | Install the `maprlogin` stub binary and create `/tmp/maprticket` so `validate_prerequisites` passes |
| 3 | Set Airflow Variables (`auth_method`, `mapr_user`, `mapr_ticketfile_location`, `migration_distcp_mappers`) via REST API |
| 4 | Build an Excel migration config covering the test scenarios and upload it to S3 |
| 5 | Trigger a `source_to_s3_migration` DAG run via the Airflow REST API |
| 6 | Poll the run until it reaches a terminal state and print task summary |
| 7 | Render the DAG's HTML migration report inline |

### Prerequisites

- `kubectl` configured with access to the `hadoop-ssh` namespace (the same kubeconfig you use to `kubectl exec`).
- Airflow webserver reachable from JH and credentials with permission to write Variables and trigger DAGs.
- JH Spark session has S3 write access for the bucket where the Excel config will be uploaded.

Reference: [MapR Edge Node Setup Guide](https://www.notion.so/MapR-Edge-Node-Setup-Guide-351984e820aa80e29cd9fa0d7fd127d9).

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────────
import os

# --- Edge node (SSH — same connection Airflow uses via `cluster_edge_ssh`) ---
# Example: "hadoop-edge-ssh.<edge-pod-namespace>.svc.cluster.local"
SSH_HOST     = os.environ.get("EDGE_SSH_HOST", "")
SSH_PORT     = int(os.environ.get("EDGE_SSH_PORT", "22"))
SSH_USER     = os.environ.get("EDGE_SSH_USER", "root")
SSH_PASSWORD = os.environ.get("EDGE_SSH_PASSWORD", "")
MAPR_USER       = "root"
MAPR_TICKETFILE = "/tmp/maprticket"

# --- Airflow REST API ---
# In-cluster URL bypasses Keycloak (which only fronts the public Ingress).
# The Airflow webserver behind it uses FAB auth manager → JWT via /auth/token.
# Example: "http://airflow-api-server.<tenant-namespace>.svc.cluster.local:8080"
AIRFLOW_BASE_URL = os.environ.get("AIRFLOW_BASE_URL", "")
AIRFLOW_USERNAME = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD = os.environ.get("AIRFLOW_PASSWORD", "")
DAG_ID           = "source_to_s3_migration"

# --- Excel config destination on S3 ---
# Must be a path the DAG's Spark session can read via s3a://.
# TENANT_PREFIX example: f"s3a://{BUCKET}/<tenant>"
BUCKET            = os.environ.get("S3_BUCKET", "")
TENANT_PREFIX     = os.environ.get("S3_TENANT_PREFIX", "")
EXCEL_S3_PATH     = f"{TENANT_PREFIX}/configs/wf201_e2e_test.xlsx"

# --- Tracking + report (must match DAG's MIGRATION_TRACKING_DATABASE / MIGRATION_REPORT_LOCATION) ---
TRACKING_DB     = "migration_tracking"
REPORT_LOCATION = f"{TENANT_PREFIX}/migration_reports"

# --- DistCp tuning required by the edge-node pseudo-distributed cluster ---
# Per the Notion guide: mappers must be 1 and DAG distcp must use mapreduce.framework.name=local.
# Mappers is set as an Airflow Variable below; the framework=local flag must already be in the DAG code.
DISTCP_MAPPERS = "1"

print("Configuration loaded.")
print(f"  SSH target     : {SSH_USER}@{SSH_HOST}:{SSH_PORT}")
print(f"  Airflow URL    : {AIRFLOW_BASE_URL or '(unset — set AIRFLOW_BASE_URL)'}")
print(f"  Airflow user   : {AIRFLOW_USERNAME or '(unset — set AIRFLOW_USERNAME/PASSWORD env vars)'}")
print(f"  DAG ID         : {DAG_ID}")
print(f"  Excel S3 path  : {EXCEL_S3_PATH}")
print(f"  Report location: {REPORT_LOCATION}")


In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────
import re
import subprocess
import sys

import requests

try:
    import paramiko
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "paramiko"])
    import paramiko

# Login-shell init scripts on this pod emit these lines on every SSH session.
# They are not errors and not specific to anything we run.
_LOGIN_NOISE = re.compile(
    r"^(mesg: ttyname failed: Inappropriate ioctl for device"
    r"|ls: cannot access '/opt/spark/lib/spark-assembly-\*\.jar': No such file or directory)$"
)


def _strip_login_noise(text):
    return "\n".join(line for line in text.splitlines() if not _LOGIN_NOISE.match(line))


def _ssh_connect():
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(
        hostname=SSH_HOST, port=SSH_PORT,
        username=SSH_USER, password=SSH_PASSWORD,
        timeout=30, allow_agent=False, look_for_keys=False,
    )
    return client


def pod_exec(script, check=True, timeout=600):
    """Run a bash script in the edge pod via SSH login shell. Prints stdout/stderr."""
    preview = script if len(script) <= 200 else script[:197] + "..."
    print(f"$ ({SSH_HOST}) {preview}")
    client = _ssh_connect()
    try:
        stdin, stdout, stderr = client.exec_command("bash -l", timeout=timeout)
        stdin.write(script + "\n")
        stdin.channel.shutdown_write()
        out = stdout.read().decode()
        err = stderr.read().decode()
        rc = stdout.channel.recv_exit_status()
    finally:
        client.close()
    out = _strip_login_noise(out)
    err = _strip_login_noise(err)
    if out.strip():
        print(out.rstrip())
    if err.strip():
        print(err.rstrip())
    if check and rc != 0:
        raise RuntimeError(f"pod exec exited {rc}")


_AIRFLOW_TOKEN = {"value": None}


def _airflow_token():
    """Get (and cache) a JWT from Airflow's FAB auth manager."""
    if _AIRFLOW_TOKEN["value"]:
        return _AIRFLOW_TOKEN["value"]
    if not AIRFLOW_USERNAME or not AIRFLOW_PASSWORD:
        raise RuntimeError("Set AIRFLOW_USERNAME and AIRFLOW_PASSWORD env vars before calling Airflow.")
    url = AIRFLOW_BASE_URL.rstrip("/") + "/auth/token"
    resp = requests.post(url, json={"username": AIRFLOW_USERNAME, "password": AIRFLOW_PASSWORD}, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} -> {resp.status_code}: {resp.text[:300]}")
    _AIRFLOW_TOKEN["value"] = resp.json()["access_token"]
    return _AIRFLOW_TOKEN["value"]


def airflow_request(method, path, **kwargs):
    """Call the Airflow v2 REST API with a JWT bearer token."""
    headers = {"Authorization": f"Bearer {_airflow_token()}", **kwargs.pop("headers", {})}
    url = AIRFLOW_BASE_URL.rstrip("/") + path
    resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if resp.status_code == 401:
        # Token may have expired; refresh once.
        _AIRFLOW_TOKEN["value"] = None
        headers["Authorization"] = f"Bearer {_airflow_token()}"
        resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if not resp.ok:
        raise RuntimeError(f"{method} {url} -> {resp.status_code}: {resp.text[:500]}")
    return resp.json() if resp.content else {}


print("Helpers ready.")

## Step 0 — Verify the edge pod is reachable

Connects via SSH (which proves port 22 is up) and checks that the Hive metastore (9083) and HiveServer2 (10000) ports are also listening. If 9083/10000 are missing, the pod was probably just restarted — wait ~90s and re-run.

In [ ]:
# SSH connect proves port 22 is up. Then check Hive metastore (9083) + HiveServer2 (10000).
pod_exec("hostname && uptime")
pod_exec("ss -tlnp 2>/dev/null | grep -E ':(22|9083|10000) ' || netstat -tlnp 2>/dev/null | grep -E ':(22|9083|10000) '")

## Step 1 — Populate HDFS + Hive metastore with test data

Runs the bundled `/setup-test-data.sh`. Creates 4 databases (`sales_db`, `hr_db`, `analytics_db`, `logs_db`) with 10 tables across them — partitioned, non-partitioned, empty, unregistered partitions, mixed delimiters. Idempotent.

In [ ]:
pod_exec("bash /setup-test-data.sh")

## Step 2 — Install MapR `maprlogin` stub and ticket file

The DAG's `validate_prerequisites` task runs `maprlogin print | grep -q <user>` over SSH. The pod is not a real MapR node, so we install a stub binary that always reports a valid ticket for `root` and persist `MAPR_TICKETFILE_LOCATION` so it survives across SSH sessions.

In [ ]:
MAPR_SETUP_SCRIPT = r'''
# Note: deliberately no `set -o pipefail` — `grep -q` closes the pipe on first match,
# which sends SIGPIPE (exit 141) to maprlogin and would fail the smoke-test pipeline.
# The DAG's validate_prerequisites also runs without pipefail for the same reason.
set -eu

cat > /usr/local/bin/maprlogin << 'EOF'
#!/bin/bash
if [ "$1" = "print" ]; then
  echo "MapR credentials (UID 0) for user: root"
  echo "  created: $(date)"
  echo "  expires: $(date -d '+7 days' 2>/dev/null || date)"
  echo "  cluster: test-cluster"
  exit 0
fi
echo "maprlogin: unknown command '$1'"
exit 1
EOF
chmod +x /usr/local/bin/maprlogin

mkdir -p /tmp
cat > /tmp/maprticket << 'EOF'
MAPR_TICKET
cluster=test-cluster
user=root
uid=0
created=0
expires=9999999999
EOF

# Persist for Airflow's SSH sessions (DAG sources ~/.profile for MapR cluster type).
for rc in /root/.profile /root/.bashrc; do
  grep -q MAPR_TICKETFILE_LOCATION "$rc" || echo 'export MAPR_TICKETFILE_LOCATION=/tmp/maprticket' >> "$rc"
done

if maprlogin print 2>/dev/null | grep -q "root"; then
  echo "TICKET CHECK: PASSED"
else
  echo "TICKET CHECK: FAILED"
  exit 1
fi
'''

pod_exec(MAPR_SETUP_SCRIPT)

## Step 2b — Expose the S3A connector on the `hadoop fs` classpath

The edge pod's `hadoop fs` shell does **not** auto-load `share/hadoop/tools/lib/*`, where `hadoop-aws.jar` (containing `S3AFileSystem`) lives. `hadoop distcp` works because it is a Hadoop *tool* and pulls that dir in automatically; the FsShell does not.

The DAG's `calculate_s3_metrics_hadoop` invokes `hadoop fs -test -d` / `-ls -R` / `-du -s` against `s3a://…` paths. Without this step those calls die with `ClassNotFoundException: org.apache.hadoop.fs.s3a.S3AFileSystem` and the `2>/dev/null` masks in the DAG turn the failures into silent zeros — every table validates as "S3 has 0 files" and the run reports false FAILs even though distcp copied successfully.

Idempotently appends a `HADOOP_CLASSPATH` export to `~/.profile` (which the DAG's MapR-mode SSH sources via `_login_shell`).

In [ ]:
pod_exec(r"""
PROFILE="$HOME/.profile"
LINE='export HADOOP_CLASSPATH="${HADOOP_CLASSPATH:+$HADOOP_CLASSPATH:}${HADOOP_HOME:-/opt/hadoop}/share/hadoop/tools/lib/*"'
if grep -qF "share/hadoop/tools/lib" "$PROFILE" 2>/dev/null; then
    echo "already present in $PROFILE"
else
    echo "$LINE" >> "$PROFILE"
    echo "appended to $PROFILE"
fi
""")

## Step 3 — Configure Airflow Variables

The DAG reads its runtime config from Airflow Variables (see `migrator_utils/migrations/shared.py::get_config`). Four are critical for this test:

| Key | Value | Purpose |
|---|---|---|
| `auth_method` | `mapr` | Picks the `maprlogin` branch in `validate_prerequisites` |
| `mapr_user` | `root` | Username the DAG greps for in `maprlogin print` output |
| `mapr_ticketfile_location` | `/tmp/maprticket` | Path passed to the SSH session env |
| `migration_distcp_mappers` | `1` | Forces single-mapper distcp (pseudo-distributed cluster cannot launch 50 YARN containers) |

In [ ]:
VARIABLES = {
    "auth_method": "mapr",
    "mapr_user": MAPR_USER,
    "mapr_ticketfile_location": MAPR_TICKETFILE,
    "migration_distcp_mappers": DISTCP_MAPPERS,
    # Keep MapR — it's the DAG default and emits no committer flags. The edge node is
    # Hadoop 2.7.7; cluster_type=HDP would emit S3ACommitterFactory which only exists in
    # Hadoop 3.1+, causing distcp to fail at job init with ClassNotFoundException.
    "cluster_type": "MapR",
}

for key, value in VARIABLES.items():
    body = {"key": key, "value": value}
    try:
        airflow_request("PATCH", f"/api/v2/variables/{key}", json=body)
        action = "updated"
    except RuntimeError as e:
        if "404" in str(e):
            airflow_request("POST", "/api/v2/variables", json=body)
            action = "created"
        else:
            raise
    print(f"  {action}: {key} = {value}")

## Step 4 — Build and upload the Excel migration config

Generates an Excel workbook covering the partition-filter scenarios from the setup guide and uploads it to S3 via Spark's Hadoop FS (same path the DAG will read with `binaryFile`). Each row maps to one `(database, table, partition_filter)` discovery group.

| Source DB | Tables | Filter | Tests |
|---|---|---|---|
| `sales_db` | `orders` | _(blank)_ | All partitions, multi-partition table |
| `hr_db` | `*` | _(blank)_ | Wildcard table expansion |
| `analytics_db` | `events` | `region=US/*` | Wildcard partition filter |
| `analytics_db` | `sessions` | _(blank)_ | Unregistered partitions |
| `logs_db` | `app_logs` | `dt>=2024-01-15` | `>=` filter — the WF-201 fix path |

In [ ]:
from io import BytesIO

import pandas as pd
from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("trigger-mapr-to-s3-e2e") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

java_import(spark._jvm, "org.apache.hadoop.fs.*")

rows = [
    {"database": "sales_db",     "table": "orders",   "partition_filter": "",                "dest_database": "sales_db_copy",     "bucket": BUCKET, "endpoint": ""},
    {"database": "hr_db",        "table": "*",        "partition_filter": "",                "dest_database": "hr_db_copy",        "bucket": BUCKET, "endpoint": ""},
    {"database": "analytics_db", "table": "events",   "partition_filter": "region=US/*",     "dest_database": "analytics_db_copy", "bucket": BUCKET, "endpoint": ""},
    {"database": "analytics_db", "table": "sessions", "partition_filter": "",                "dest_database": "analytics_db_copy", "bucket": BUCKET, "endpoint": ""},
    {"database": "logs_db",      "table": "app_logs", "partition_filter": "dt>=2024-01-15",  "dest_database": "logs_db_copy",      "bucket": BUCKET, "endpoint": ""},
]
df = pd.DataFrame(rows)

buf = BytesIO()
df.to_excel(buf, index=False, engine="openpyxl")
xlsx_bytes = buf.getvalue()

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI(EXCEL_S3_PATH), spark._jsc.hadoopConfiguration()
)
p = spark._jvm.org.apache.hadoop.fs.Path(EXCEL_S3_PATH)
out = fs.create(p, True)
try:
    out.write(xlsx_bytes)
finally:
    out.close()

print(f"Uploaded {len(xlsx_bytes)} bytes to {EXCEL_S3_PATH}")
df

## Step 4b — Wipe destination S3 structure (pre-run reset)

Recursively deletes every dest-database S3 prefix (`s3a://{BUCKET}/{dest_database}/` for each `dest_database` referenced in the Excel rows above) and the report directory. Ensures the next DAG run starts against an empty S3 surface.

Hive metadata is not touched here — use Step 8 (per-run tracking cleanup) or Step 9 (drop tracking DB) if you also need to reset those.

In [ ]:
from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.*")

dest_dbs = sorted({r["dest_database"] for r in rows})
targets = [f"s3a://{BUCKET}/{db}" for db in dest_dbs] + [REPORT_LOCATION]

FS = spark._jvm.org.apache.hadoop.fs.FileSystem
Path = spark._jvm.org.apache.hadoop.fs.Path
URI = spark._jvm.java.net.URI
hconf = spark._jsc.hadoopConfiguration()

for p in targets:
    try:
        fs = FS.get(URI(p), hconf)
        jp = Path(p)
        if fs.exists(jp):
            fs.delete(jp, True)
            print(f"  deleted: {p}")
        else:
            print(f"  skip (not present): {p}")
    except Exception as e:
        print(f"  ERROR deleting {p}: {e}")

## Step 5 — Trigger a DAG run

POSTs to `/api/v1/dags/source_to_s3_migration/dagRuns` with the Excel path as a conf override. Returns the `dag_run_id` used in Step 6.

In [ ]:
from datetime import datetime, timezone

logical_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
run_id = f"manual_e2e_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

trigger_body = {
    "dag_run_id": run_id,
    "logical_date": logical_date,
    "conf": {"excel_file_path": EXCEL_S3_PATH},
}

resp = airflow_request("POST", f"/api/v2/dags/{DAG_ID}/dagRuns", json=trigger_body)
print(f"Triggered: dag_run_id={resp.get('dag_run_id')} state={resp.get('state')}")
print(f"  excel_file_path = {EXCEL_S3_PATH}")
DAG_RUN_ID = resp["dag_run_id"]

## Step 6 — Monitor the run

Polls `/api/v1/dags/{dag_id}/dagRuns/{run_id}` every 15s until the run reaches a terminal state (`success`, `failed`). Then prints per-task state. The full migration takes a few minutes against this small dataset.

In [ ]:
import time

TERMINAL = {"success", "failed"}
POLL_INTERVAL = 15
TIMEOUT_SECS = 60 * 30

deadline = time.time() + TIMEOUT_SECS
last_state = None
while time.time() < deadline:
    run = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}")
    state = run.get("state")
    if state != last_state:
        print(f"[{time.strftime('%H:%M:%S')}] state = {state}")
        last_state = state
    if state in TERMINAL:
        break
    time.sleep(POLL_INTERVAL)
else:
    print(f"Timed out after {TIMEOUT_SECS}s waiting for terminal state.")

tasks = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}/taskInstances")
print("\nTask summary:")
print(f"  {'task_id':<40} {'state':<10} {'try':<5} duration")
print(f"  {'-'*40} {'-'*10} {'-'*5} {'-'*10}")
for t in sorted(tasks.get("task_instances", []), key=lambda x: (x.get("start_date") or "")):
    print(f"  {t.get('task_id',''):<40} {str(t.get('state','')):<10} {str(t.get('try_number','')):<5} {t.get('duration')}")

## Step 7 — Render the HTML migration report

The DAG's `generate_report` task writes an HTML summary to `{REPORT_LOCATION}/{run_id}_report.html`. Looks up the matching `run_id` from `migration_tracking.migration_runs` (filtered by our `DAG_RUN_ID` so we always grab this run's report, not the latest unrelated one) and renders the report inline.

In [ ]:
from IPython.display import HTML, display

try:
    rows = spark.sql(f"""
        SELECT run_id FROM {TRACKING_DB}.migration_runs
        WHERE dag_run_id = '{DAG_RUN_ID}'
        ORDER BY started_at DESC LIMIT 1
    """).collect()

    if not rows:
        print(f"No migration run found in {TRACKING_DB}.migration_runs for dag_run_id={DAG_RUN_ID}.")
    else:
        rid = rows[0]["run_id"]
        report_path = f"{REPORT_LOCATION}/{rid}_report.html"
        print(f"Reading report: {report_path}\n")

        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
            spark._jvm.java.net.URI(report_path), spark._jsc.hadoopConfiguration()
        )
        p = spark._jvm.org.apache.hadoop.fs.Path(report_path)
        if not fs.exists(p):
            print(f"Report file not found at {report_path}")
            print("REPORT_LOCATION may differ from the DAG's MIGRATION_REPORT_LOCATION — check the config cell.")
        else:
            reader = spark._jvm.java.io.BufferedReader(
                spark._jvm.java.io.InputStreamReader(fs.open(p), "UTF-8")
            )
            lines = []
            line = reader.readLine()
            while line is not None:
                lines.append(line)
                line = reader.readLine()
            reader.close()
            display(HTML("\n".join(lines)))

except Exception as e:
    print(f"Could not load report: {e}")

## Step 8 — Per-run cleanup (optional)

Removes artifacts of the current `DAG_RUN_ID` so reruns don't pile up:

- migrated S3 data at every `dest_location` recorded in `migration_table_status` for this run
- the report HTML at `{REPORT_LOCATION}/{run_id}_report.html`
- this run's rows in `migration_runs` and `migration_table_status`

Does **not** touch the test Hive DBs on the edge node, the Excel config, or the Airflow Variables.

In [ ]:
try:
    rows = spark.sql(f"""
        SELECT run_id FROM {TRACKING_DB}.migration_runs
        WHERE dag_run_id = '{DAG_RUN_ID}'
    """).collect()

    if not rows:
        print(f"No tracking row for dag_run_id={DAG_RUN_ID}; nothing to clean.")
    else:
        rid = rows[0]["run_id"]
        print(f"Cleaning artifacts for run_id={rid}")

        dest_paths = [r["dest_location"] for r in spark.sql(f"""
            SELECT DISTINCT dest_location FROM {TRACKING_DB}.migration_table_status
            WHERE run_id = '{rid}' AND dest_location IS NOT NULL
        """).collect()]

        targets = list(dest_paths) + [f"{REPORT_LOCATION}/{rid}_report.html"]

        FS = spark._jvm.org.apache.hadoop.fs.FileSystem
        Path = spark._jvm.org.apache.hadoop.fs.Path
        URI = spark._jvm.java.net.URI
        hconf = spark._jsc.hadoopConfiguration()

        for p in targets:
            try:
                fs = FS.get(URI(p), hconf)
                jp = Path(p)
                if fs.exists(jp):
                    fs.delete(jp, True)
                    print(f"  deleted: {p}")
                else:
                    print(f"  skip (not found): {p}")
            except Exception as e:
                print(f"  ERROR deleting {p}: {e}")

        spark.sql(f"DELETE FROM {TRACKING_DB}.migration_table_status WHERE run_id = '{rid}'")
        spark.sql(f"DELETE FROM {TRACKING_DB}.migration_runs        WHERE run_id = '{rid}'")
        print(f"Deleted tracking rows for run_id={rid}.")
except Exception as e:
    print(f"Cleanup failed: {e}")

## Step 9 — Drop the tracking database (destructive)

**DESTRUCTIVE — affects every previous migration run, not just this one.** Use only when the tracking-DB schema has drifted (e.g. after a DAG schema change) and you want the next DAG run to recreate the tables from scratch via `init_tracking_tables`.

Drops both Iceberg tables with `PURGE` so metadata + data files are removed, then drops the database, then best-effort wipes the underlying S3 location.

In [ ]:
try:
    loc = None
    try:
        for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {TRACKING_DB}").collect():
            if r[0].lower() in ("location", "location_uri"):
                loc = r[1]
                break
    except Exception:
        pass

    for tbl in ("migration_table_status", "migration_runs"):
        try:
            spark.sql(f"DROP TABLE IF EXISTS {TRACKING_DB}.{tbl} PURGE")
            print(f"  dropped table: {TRACKING_DB}.{tbl}")
        except Exception as e:
            print(f"  ERROR dropping {TRACKING_DB}.{tbl}: {e}")

    spark.sql(f"DROP DATABASE IF EXISTS {TRACKING_DB} CASCADE")
    print(f"Dropped database: {TRACKING_DB}")

    if loc:
        try:
            FS = spark._jvm.org.apache.hadoop.fs.FileSystem
            Path = spark._jvm.org.apache.hadoop.fs.Path
            URI = spark._jvm.java.net.URI
            hconf = spark._jsc.hadoopConfiguration()
            fs = FS.get(URI(loc), hconf)
            jp = Path(loc)
            if fs.exists(jp):
                fs.delete(jp, True)
                print(f"  wiped storage: {loc}")
            else:
                print(f"  storage already gone: {loc}")
        except Exception as e:
            print(f"  storage wipe failed for {loc}: {e}")
except Exception as e:
    print(f"Drop failed: {e}")

## Inspect source tables on the edge node (one-off)

Lists every Hive `.db` directory under `/user/hive/warehouse/` recursively. Useful for sanity-checking what DistCp will see as its source.

In [ ]:
pod_exec(r"""
set -e
echo "=== Hive warehouse top-level ==="
hdfs dfs -ls /user/hive/warehouse/ || true

for db_dir in $(hdfs dfs -ls /user/hive/warehouse/ 2>/dev/null | awk '/\.db$/ {print $NF}'); do
    echo
    echo "=== $db_dir (recursive) ==="
    hdfs dfs -ls -R "$db_dir" || true
done
""")

## Inspect S3 destinations for the current run (one-off)

Reads `dest_location` from `migration_table_status` for the current `DAG_RUN_ID` and lists each path recursively via Spark's Hadoop FileSystem. Uses S3A directly so no `hadoop fs` CLI needed in the pod.

In [ ]:
rid_rows = spark.sql(f"""
    SELECT run_id FROM {TRACKING_DB}.migration_runs
    WHERE dag_run_id = '{DAG_RUN_ID}'
    ORDER BY started_at DESC LIMIT 1
""").collect()

if not rid_rows:
    print(f"No tracking row for dag_run_id={DAG_RUN_ID}.")
else:
    rid = rid_rows[0]["run_id"]
    rows = spark.sql(f"""
        SELECT source_database, source_table, dest_location
        FROM {TRACKING_DB}.migration_table_status
        WHERE run_id = '{rid}'
        ORDER BY source_database, source_table
    """).collect()

    FS = spark._jvm.org.apache.hadoop.fs.FileSystem
    Path = spark._jvm.org.apache.hadoop.fs.Path
    URI = spark._jvm.java.net.URI
    hconf = spark._jsc.hadoopConfiguration()

    def list_recursive(fs, path):
        out = []
        it = fs.listFiles(path, True)
        while it.hasNext():
            st = it.next()
            out.append((st.getPath().toString(), st.getLen()))
        return out

    for r in rows:
        loc = r["dest_location"]
        header = f"=== {r['source_database']}.{r['source_table']} -> {loc} ==="
        print(f"\n{header}")
        if not loc:
            print("  (no dest_location recorded)")
            continue
        try:
            fs = FS.get(URI(loc), hconf)
            jp = Path(loc)
            if not fs.exists(jp):
                print("  (path does not exist on S3)")
                continue
            files = list_recursive(fs, jp)
            if not files:
                print("  (path exists but contains 0 files)")
            else:
                total = 0
                for p, size in files:
                    print(f"  {size:>10}  {p}")
                    total += size
                print(f"  ---- {len(files)} file(s), {total} bytes total")
        except Exception as e:
            print(f"  ERROR: {e}")